# NB12 — PANDA Out-of-Fold Metrics

Reads the seed-averaged OOF predictions saved by NB11 and computes macro one-vs-rest AUROC across all six ISUP grades plus binarized AUROC and AUPRC at thresholds ≥1, ≥2, ≥3, ≥4, ≥5. Stratifies the same metrics by `data_provider` (Karolinska vs Radboud) for the within-provider Figure 4F.

Outputs `metrics_auc.json` and `metrics_auc_by_provider.csv` for NB13.

In [ ]:
import os, json
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.metrics import roc_auc_score, average_precision_score

WORKSPACE = Path(os.environ.get('WORKSPACE', './workspace'))
PANDA_OOF_CANDIDATES = [
    WORKSPACE / 'results' / 'panda_mil'     / 'oof_ensemble.csv',
    WORKSPACE / 'results' / 'panda_mil'     / 'oof_predictions.csv',
    WORKSPACE / 'results' / 'panda_mil'     / 'oof.csv',
    WORKSPACE / 'results' / 'panda_mil_088' / 'oof_ensemble.csv',
]

OOF_CSV = next((p for p in PANDA_OOF_CANDIDATES if p.exists()), None)
if OOF_CSV is None:
    print('[WARN] no PANDA OOF file found; run NB11 first')
    raise SystemExit(0)

print(f'reading OOF from {OOF_CSV}')
df = pd.read_csv(OOF_CSV)
assert 'true_isup' in df.columns, 'expected true_isup column'
y_true = df['true_isup'].astype(int).values
num_classes = int(max(y_true.max(), 5) + 1)

prob_cols = sorted([c for c in df.columns if c.startswith('prob_')], key=lambda c: int(c.split('_')[-1]))
logit_cols = sorted([c for c in df.columns if c.startswith('logit_')], key=lambda c: int(c.split('_')[-1]))

if prob_cols:
    P = df[prob_cols].to_numpy(float)
    s = P.sum(axis=1, keepdims=True); s[s == 0] = 1.0
    P = P / s
elif logit_cols:
    Z = df[logit_cols].to_numpy(float)
    Z = Z - Z.max(axis=1, keepdims=True)
    P = np.exp(Z); P /= P.sum(axis=1, keepdims=True)
else:
    raise RuntimeError('neither prob_* nor logit_* columns found')

def safe_macro_ovr(y, P_mat):
    try:
        return float(roc_auc_score(y, P_mat, multi_class='ovr', average='macro'))
    except Exception:
        return float('nan')

def thresh_scores(y, P_mat, thr):
    y_bin = (y >= thr).astype(int)
    s_bin = P_mat[:, thr:].sum(axis=1)
    return y_bin, s_bin

def bin_metrics(y_bin, s_bin):
    return float(roc_auc_score(y_bin, s_bin)), float(average_precision_score(y_bin, s_bin))

metrics = {'macro_auroc_ovr': safe_macro_ovr(y_true, P), 'thresholds': {}}
for t in [1, 2, 3, 4, 5]:
    yb, sb = thresh_scores(y_true, P, t)
    auroc, aupr = bin_metrics(yb, sb)
    metrics['thresholds'][f'>={t}'] = {
        'auroc': auroc, 'auprc': aupr, 'pos_rate': float(yb.mean()),
    }

by_prov = []
prov_col = 'data_provider' if 'data_provider' in df.columns else None
if prov_col:
    for prov, dsub in df.groupby(prov_col):
        y_sub = dsub['true_isup'].astype(int).values
        P_sub = dsub[prob_cols].to_numpy(float) if prob_cols else None
        if P_sub is not None:
            s = P_sub.sum(axis=1, keepdims=True); s[s == 0] = 1.0
            P_sub = P_sub / s
        row = {
            'provider': prov,
            'macro_auroc_ovr': safe_macro_ovr(y_sub, P_sub),
            'n': int(len(dsub)),
        }
        for t in [1, 2, 3, 4, 5]:
            yb, sb = thresh_scores(y_sub, P_sub, t)
            auroc, aupr = bin_metrics(yb, sb)
            row[f'AUROC_>={t}'] = auroc
            row[f'AUPRC_>={t}'] = aupr
        by_prov.append(row)

out_dir = OOF_CSV.parent
(out_dir / 'figures').mkdir(exist_ok=True)
(out_dir / 'metrics_auc.json').write_text(json.dumps(metrics, indent=2))
if by_prov:
    pd.DataFrame(by_prov).to_csv(out_dir / 'metrics_auc_by_provider.csv', index=False)

print(f'\nPANDA AUROC/AUPRC ({num_classes}-class)')
print(f'  macro AUROC (OvR): {metrics["macro_auroc_ovr"]:.4f}')
for t in [1, 2, 3, 4, 5]:
    m = metrics['thresholds'][f'>={t}']
    print(f'  ISUP ≥{t}: AUROC {m["auroc"]:.4f} | AUPRC {m["auprc"]:.4f} | prevalence {m["pos_rate"]*100:.1f}%')
if by_prov:
    print('\nPer-provider:')
    for row in by_prov:
        print(f'  {row["provider"]:10s} | n={row["n"]:4d} | macro AUROC {row["macro_auroc_ovr"]:.4f}')
print(f'\n[OK] saved: {out_dir / "metrics_auc.json"}')
print('NB12 complete. Next: NB13 (manuscript figures).')